# Aviothic Breast Cancer Detection - Advanced Training

This notebook demonstrates advanced training techniques including transfer learning and hyperparameter tuning.

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import DataLoader
import mlflow
import mlflow.pytorch

ModuleNotFoundError: No module named 'mlflow'

In [ ]:
# Use transfer learning with ResNet
class AdvancedBreastCancerClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super(AdvancedBreastCancerClassifier, self).__init__()
        self.resnet = models.resnet50(pretrained=True)
        
        # Freeze early layers
        for param in self.resnet.parameters():
            param.requires_grad = False
            
        # Replace classifier
        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
        
    def forward(self, x):
        return self.resnet(x)

In [ ]:
# Hyperparameter tuning with MLFlow
def train_with_mlflow(learning_rate=0.001, batch_size=32, epochs=10):
    with mlflow.start_run():
        # Log parameters
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("epochs", epochs)
        
        # Initialize model
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = AdvancedBreastCancerClassifier().to(device)
        
        # Loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        
        # Training loop
        for epoch in range(epochs):
            # Training code here
            # For demonstration, we'll just log dummy metrics
            train_loss = 0.5 - (epoch * 0.05)  # Dummy decreasing loss
            val_accuracy = 0.7 + (epoch * 0.03)  # Dummy increasing accuracy
            
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_accuracy", val_accuracy, step=epoch)
            
        # Save model
        mlflow.pytorch.log_model(model, "model")
        
        return model

In [ ]:
# Run hyperparameter tuning
learning_rates = [0.001, 0.01, 0.0001]
batch_sizes = [16, 32, 64]

best_model = None
best_params = {}

for lr in learning_rates:
    for bs in batch_sizes:
        print(f"Training with LR: {lr}, BS: {bs}")
        model = train_with_mlflow(learning_rate=lr, batch_size=bs, epochs=5)